# Capstone Two: Pre-processing and Training Data Development

## Zillow Home Value Index (ZHVI) Forecasting

The goal of this notebook is to prepare the Zillow Home Value Index dataset for machine learning modeling. This includes:

- Handling missing values
- Creating forecasting features
- Scaling numeric variables
- Splitting data into training and testing sets

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

## Load Dataset

In [2]:
df = pd.read_csv("data/raw/zhvi.csv")

df.head()

,Unnamed: 0,Florida,California,Texas,New York,Virginia,Arizona,Massachusetts,Utah,the District of Columbia,...,North Dakota,Louisiana,Wyoming,Arkansas,Oklahoma,Kentucky,Alaska,Nebraska,Delaware,South Dakota
0,2000-01-01,106839.437513,185932.000267,111831.965571,149637.593311,122778.084361,134711.717076,192670.172342,166397.340937,158963.698803,...,NaN,108000.701408,NaN,87413.751613,82185.060767,87953.050874,133100.235315,112888.886116,146721.635463,105043.280158
1,2000-02-01,107070.861680,186556.973217,111892.737950,150170.450133,123095.788877,134955.719599,193313.433955,166488.507251,159483.634688,...,NaN,108209.025004,NaN,87530.575926,82218.962313,87998.460603,133251.409063,113069.343667,146949.900655,104820.097891
2,2000-03-01,107351.393901,187400.839738,111922.213506,150683.410757,123432.630242,135269.755328,194078.813769,166631.638154,160221.370093,...,NaN,108370.909948,NaN,87663.122946,82376.988281,88081.081745,133402.503748,113236.594627,147234.111366,104507.770821
3,2000-04-01,107921.643751,189232.622338,112068.503303,151786.649610,124069.148395,135910.998187,195586.289264,166990.756067,161803.226607,...,NaN,108735.870458,NaN,87946.132255,82664.745283,88286.402842,133579.702498,113496.218868,147884.094822,104086.787429
4,2000-05-01,108536.596763,191361.793669,112163.242870,152928.107451,124683.636950,136637.856037,197218.057247,168292.216995,163558.794607,...,NaN,109122.542409,NaN,88225.267260,83051.247688,88559.280277,133846.461332,113627.876782,148625.429884,103911.646036


## Convert Date Column

In [3]:
df['Date'] = pd.to_datetime(df['Unnamed: 0'])

df.drop(columns=['Unnamed: 0'], inplace=True)

df.set_index('Date', inplace=True)

df.head()

,Florida,California,Texas,New York,Virginia,Arizona,Massachusetts,Utah,the District of Columbia,Washington,...,North Dakota,Louisiana,Wyoming,Arkansas,Oklahoma,Kentucky,Alaska,Nebraska,Delaware,South Dakota
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,106839.437513,185932.000267,111831.965571,149637.593311,122778.084361,134711.717076,192670.172342,166397.340937,158963.698803,171436.108039,...,NaN,108000.701408,NaN,87413.751613,82185.060767,87953.050874,133100.235315,112888.886116,146721.635463,105043.280158
2000-02-01,107070.861680,186556.973217,111892.737950,150170.450133,123095.788877,134955.719599,193313.433955,166488.507251,159483.634688,171822.806052,...,NaN,108209.025004,NaN,87530.575926,82218.962313,87998.460603,133251.409063,113069.343667,146949.900655,104820.097891
2000-03-01,107351.393901,187400.839738,111922.213506,150683.410757,123432.630242,135269.755328,194078.813769,166631.638154,160221.370093,172217.272453,...,NaN,108370.909948,NaN,87663.122946,82376.988281,88081.081745,133402.503748,113236.594627,147234.111366,104507.770821
2000-04-01,107921.643751,189232.622338,112068.503303,151786.649610,124069.148395,135910.998187,195586.289264,166990.756067,161803.226607,173013.406497,...,NaN,108735.870458,NaN,87946.132255,82664.745283,88286.402842,133579.702498,113496.218868,147884.094822,104086.787429
2000-05-01,108536.596763,191361.793669,112163.242870,152928.107451,124683.636950,136637.856037,197218.057247,168292.216995,163558.794607,173882.336749,...,NaN,109122.542409,NaN,88225.267260,83051.247688,88559.280277,133846.461332,113627.876782,148625.429884,103911.646036


## Missing Value Analysis

Several states contain missing values. Missing values will be handled before model development to ensure a complete dataset.

In [4]:
df.isnull().sum().sort_values(ascending=False).head(10)

North Dakota     108
Montana           61
New Mexico        27
Wyoming           27
South Dakota       1
West Virginia      1
Idaho              1
Alaska             1
Arizona            1
Tennessee          0
dtype: int64

In [5]:
df = df.interpolate()

In [6]:
df.isnull().sum().sort_values(ascending=False).head()

North Dakota    108
Montana          61
New Mexico       27
Wyoming          27
Florida           0
dtype: int64

## Dummy Variables

The dataset contains only numeric home value measurements and a datetime index. No categorical variables were present after preprocessing, therefore dummy variable creation was not required.

## Feature Engineering

To support time-series forecasting, lagged home value features were created. These lag features use previous Florida home values to predict future Florida home values.

In [7]:
df['Florida_lag1'] = df['Florida'].shift(1)
df['Florida_lag2'] = df['Florida'].shift(2)
df['Florida_lag3'] = df['Florida'].shift(3)

df = df.dropna()

## Define Features and Target Variable

In [8]:
X = df[['Florida_lag1',
        'Florida_lag2',
        'Florida_lag3']]

y = df['Florida']

## Training and Testing Split

Because this project focuses on forecasting future home values, the data was split chronologically rather than randomly. This prevents future information from leaking into the training data.

In [9]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

## Feature Scaling

Home values vary significantly in magnitude across states. Standardization was applied to ensure all predictor variables are on a comparable scale for modeling.

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [11]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

## Verify Results

In [12]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(y_train.shape)
print(y_test.shape)

(164, 3)
(41, 3)
(164,)
(41,)


In [13]:
X_train_scaled_df.describe()

,Florida_lag1,Florida_lag2,Florida_lag3
count,1.640000e+02,1.640000e+02,1.640000e+02
mean,-7.798640e-16,-5.199093e-16,-1.039819e-15
std,1.003063e+00,1.003063e+00,1.003063e+00
min,-1.169306e+00,-1.184100e+00,-1.200297e+00
25%,-8.366834e-01,-8.412913e-01,-8.468383e-01
50%,-2.473011e-01,-2.409763e-01,-2.278742e-01
75%,6.582485e-01,6.826608e-01,7.040362e-01
max,3.097498e+00,3.113542e+00,3.079144e+00


## Save Processed Data

The processed training and testing datasets are saved for use during the modeling phase of the project.

In [14]:
X_train_scaled_df.to_csv("data/X_train_scaled.csv")

X_test_scaled_df.to_csv("data/X_test_scaled.csv")

y_train.to_csv("data/y_train.csv")

y_test.to_csv("data/y_test.csv")